## **설치**

In [100]:
!pip install pytorch-tabnet

## **설정**

In [101]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [102]:
## Import libaries
import pandas as pd
import numpy as np
import os
import torch
from sklearn.preprocessing import LabelEncoder
from pytorch_tabnet.tab_model import TabNetRegressor
from sklearn.metrics import r2_score

In [103]:
## Setting
base_path = '/content/drive/MyDrive/CS/'

## **데이터 불러오기**

In [104]:
## Load the data
file_path = os.path.join(base_path, 'mergedData')

train2018 = pd.read_csv(os.path.join(file_path, 'train2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2018 = pd.read_csv(os.path.join(file_path, 'valid2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2018 = pd.read_csv(os.path.join(file_path, 'test2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2019 = pd.read_csv(os.path.join(file_path, 'train2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2019 = pd.read_csv(os.path.join(file_path, 'valid2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2019 = pd.read_csv(os.path.join(file_path, 'test2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2020 = pd.read_csv(os.path.join(file_path, 'train2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2020 = pd.read_csv(os.path.join(file_path, 'valid2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2020 = pd.read_csv(os.path.join(file_path, 'test2020.csv')).sort_values(by=['date', 'gvkey'])

train2021 = pd.read_csv(os.path.join(file_path, 'train2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2021 = pd.read_csv(os.path.join(file_path, 'valid2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2021 = pd.read_csv(os.path.join(file_path, 'test2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

In [105]:
train2018.head()

,ticker,gvkey,permno,sic,exchcd,shrcd,ffi49,ret,date,hs_20_0_x,...,hs_20_10_y,hs_20_11_y,hs_20_12_y,hs_20_13_y,hs_20_14_y,hs_20_15_y,hs_20_16_y,hs_20_17_y,hs_20_18_y,hs_20_19_y
0,AVX,1072,81912,3670,1.0,11.0,37,0.101395,1997-01-31,-0.063167,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
1,ABS,1240,50032,5411,1.0,11.0,43,-0.013333,1997-01-31,-0.058076,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
2,AGREA,1468,13056,2771,3.0,11.0,8,-0.002203,1997-01-31,-0.049929,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
3,ASC,1573,44652,5411,1.0,11.0,43,0.027523,1997-01-31,-0.066656,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069
4,ADM,1722,10516,2070,1.0,11.0,2,-0.102273,1997-01-31,-0.058967,...,0.107266,0.081205,0.002767,-0.029662,-0.100859,-0.035114,0.100308,0.096447,0.032497,-0.025069


## **데이터 전처리**

### **datetime**

In [106]:
for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name = f'{df_type}{year}'
        # Ensure the date column is converted to datetime just once for the original dataframes
        if 'date' in globals()[df_name].columns:
            globals()[df_name]['date'] = pd.to_datetime(globals()[df_name]['date'])

### **ticker와 구분 Code 백업 및 X, y분리**

In [107]:
cols_to_drop_from_X = ['ret', 'date']
categorical_features = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'Year', 'Month']

def process_year_data(year):
    # 해당 연도의 모든 데이터를 불러옴
    train_df = globals()[f'train{year}']
    valid_df = globals()[f'valid{year}']
    test_df = globals()[f'test{year}']

    # Year, Month 컬럼 생성 (datetime에서 추출)
    for df in [train_df, valid_df, test_df]:
        df['Year'] = df['date'].dt.year.astype(str)
        df['Month'] = df['date'].dt.month.astype(str)

    # _info 데이터프레임 생성 (백업용)
    globals()[f'train{year}_info'] = train_df[['ticker', 'date']].copy()
    globals()[f'valid{year}_info'] = valid_df[['ticker', 'date']].copy()
    globals()[f'test{year}_info'] = test_df[['ticker', 'date']].copy()

    # X, y 분리
    train_y = train_df[['ret']]
    valid_y = valid_df[['ret']]
    test_y = test_df[['ret']]

    train_X = train_df.drop(columns=cols_to_drop_from_X)
    valid_X = valid_df.drop(columns=cols_to_drop_from_X)
    test_X = test_df.drop(columns=cols_to_drop_from_X)

    # 3. 결측치(NaN) 처리 - 딥러닝은 NaN을 허용하지 않으므로 0으로 채움
    train_X = train_X.fillna(0)
    valid_X = valid_X.fillna(0)
    test_X = test_X.fillna(0)

    # 4. Label Encoding (범주형 변수를 0, 1, 2... 정수로 변환)
    # Train/Valid/Test에 있는 모든 범주를 합쳐서 인코딩 규칙을 만듦
    cat_idxs = []
    cat_dims = []

    for col in train_X.columns:
        if col in categorical_features:
            le = LabelEncoder()
            # 모든 데이터셋의 범주를 합쳐서 학습 (Unknown 방지)
            all_values = pd.concat([train_X[col], valid_X[col], test_X[col]]).astype(str)
            le.fit(all_values)

            train_X[col] = le.transform(train_X[col].astype(str))
            valid_X[col] = le.transform(valid_X[col].astype(str))
            test_X[col] = le.transform(test_X[col].astype(str))

            # TabNet에 전달할 인덱스와 차원 크기 저장
            cat_idxs.append(train_X.columns.get_loc(col))
            cat_dims.append(len(le.classes_))
        else:
            # 수치형 변수는 float으로 변환
            train_X[col] = train_X[col].astype(float)
            valid_X[col] = valid_X[col].astype(float)
            test_X[col] = test_X[col].astype(float)

    return train_X, train_y, valid_X, valid_y, test_X, test_y, cat_idxs, cat_dims

In [108]:
data_dict = {}
for year in range(2018, 2022):
    print(f"Processing data for {year}...")
    tX, tY, vX, vY, teX, teY, c_idxs, c_dims = process_year_data(year)
    data_dict[year] = {
        'train_X': tX, 'train_y': tY,
        'valid_X': vX, 'valid_y': vY,
        'test_X': teX, 'test_y': teY,
        'cat_idxs': c_idxs, 'cat_dims': c_dims
    }

Processing data for 2018...
Processing data for 2019...
Processing data for 2020...
Processing data for 2021...


## **TabNet**


In [109]:
from sklearn.model_selection import ParameterGrid

def tunning_tabnet(year, df_train_X, df_train_y, df_valid_X, df_valid_y, cat_idxs, cat_dims):
    # 딥러닝 학습을 위해 numpy array로 변환
    X_train = df_train_X.values
    y_train = df_train_y.values.reshape(-1, 1)
    X_valid = df_valid_X.values
    y_valid = df_valid_y.values.reshape(-1, 1)

    # 간단한 Grid Search 파라미터 (속도를 위해 범위를 좁힘)
    param_grid = {
        'n_d': [8, 16],        # 모델 복잡도 (일반적으로 n_d == n_a)
        'n_a': [8, 16],
        'learning_rate': [0.02], # 학습률
        'batch_size': [1024]     # 배치 사이즈 (메모리에 따라 조절)
    }

    best_r2 = -float('inf')
    best_params = None
    best_model = None

    print(f"Starting Grid Search for {year}...")
    grid = list(ParameterGrid(param_grid))

    for i, params in enumerate(grid):
        if params['n_d'] != params['n_a']: continue # TabNet은 보통 n_d = n_a 권장

        print(f"[{i+1}/{len(grid)}] Params: {params}")

        # 모델 초기화
        clf = TabNetRegressor(
            cat_idxs=cat_idxs,
            cat_dims=cat_dims,
            cat_emb_dim=1,      # 범주형 변수 임베딩 크기
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=params['learning_rate']),
            scheduler_params=dict(step_size=10, gamma=0.9),
            scheduler_fn=torch.optim.lr_scheduler.StepLR,
            mask_type='entmax', # or 'sparsemax'
            verbose=0,
            seed=42
        )

        # 학습
        clf.fit(
            X_train=X_train, y_train=y_train,
            eval_set=[(X_valid, y_valid)],
            eval_name=['valid'],
            eval_metric=['rmse'],
            max_epochs=100,       # 최대 에폭 (필요시 증가)
            patience=10,          # Early Stopping
            batch_size=params['batch_size'],
            virtual_batch_size=128,
            num_workers=0,
            drop_last=False
        )

        # 검증 (R-squared)
        preds = clf.predict(X_valid)
        curr_r2 = r2_score(y_valid, preds)
        print(f"  Valid R2: {curr_r2:.6f}")

        if curr_r2 > best_r2:
            best_r2 = curr_r2
            best_params = params
            best_model = clf

    print(f"Best R2 for {year}: {best_r2:.6f} with {best_params}")
    return best_model, best_params

## **실행**

In [110]:
## **실행 및 예측**

final_models = {}

for year in range(2018, 2022):
    print(f"\n====== Processing Year {year} ======")
    d = data_dict[year]

    # 1. 하이퍼파라미터 튜닝
    best_model, best_params = tunning_tabnet(
        year, d['train_X'], d['train_y'], d['valid_X'], d['valid_y'],
        d['cat_idxs'], d['cat_dims']
    )

    # 2. 최종 학습 (Train + Valid 합치기)
    print(f"--- Final Training for {year} ---")
    X_final = pd.concat([d['train_X'], d['valid_X']]).values
    y_final = pd.concat([d['train_y'], d['valid_y']]).values.reshape(-1, 1)

    # 최적 파라미터로 새 모델 생성
    final_clf = TabNetRegressor(
        cat_idxs=d['cat_idxs'],
        cat_dims=d['cat_dims'],
        cat_emb_dim=1,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=best_params['learning_rate']),
        scheduler_params=dict(step_size=10, gamma=0.9),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        verbose=0,
        seed=42
    )

    final_clf.fit(
        X_train=X_final, y_train=y_final,
        eval_set=[(X_final, y_final)], # 최종 학습시는 자기 자신을 모니터링하거나 생략 가능
        max_epochs=100,
        patience=10,
        batch_size=best_params['batch_size'],
        virtual_batch_size=128,
        num_workers=0
    )

    final_models[year] = final_clf

    # 3. 예측 및 결과 저장
    test_X_np = d['test_X'].values
    pred_y = final_clf.predict(test_X_np)

    result_df = globals()[f'test{year}_info'].copy()
    result_df['pred_ret'] = pred_y
    result_df['true_ret'] = d['test_y'].values

    # 저장
    save_dir = os.path.join(base_path, 'PredData', 'TabNet')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    save_path = os.path.join(save_dir, f'result_{year}.csv')
    result_df.to_csv(save_path, index=False)
    print(f"Saved result for {year} to {save_path}")

print("\nAll Done!")


====== Processing Year 2018 ======
Starting Grid Search for 2018...
[1/4] Params: {'batch_size': 1024, 'learning_rate': 0.02, 'n_a': 8, 'n_d': 8}

Early stopping occurred at epoch 11 with best_epoch = 1 and best_valid_rmse = 0.08398


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Valid R2: -0.001151
[4/4] Params: {'batch_size': 1024, 'learning_rate': 0.02, 'n_a': 16, 'n_d': 16}

Early stopping occurred at epoch 11 with best_epoch = 1 and best_valid_rmse = 0.08398


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Valid R2: -0.001151
Best R2 for 2018: -0.001151 with {'batch_size': 1024, 'learning_rate': 0.02, 'n_a': 8, 'n_d': 8}
--- Final Training for 2018 ---
Stop training because you reached max_epochs = 100 with best_epoch = 97 and best_val_0_mse = 0.00884


...

  Valid R2: 0.031251
[4/4] Params: {'batch_size': 1024, 'learning_rate': 0.02, 'n_a': 16, 'n_d': 16}

Early stopping occurred at epoch 12 with best_epoch = 2 and best_valid_rmse = 0.11143


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


  Valid R2: 0.031251
Best R2 for 2021: 0.031251 with {'batch_size': 1024, 'learning_rate': 0.02, 'n_a': 8, 'n_d': 8}
--- Final Training for 2021 ---

Early stopping occurred at epoch 23 with best_epoch = 13 and best_val_0_mse = 0.01308


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Saved result for 2021 to /content/drive/MyDrive/CS/PredData/TabNet/result_2021.csv

All Done!
